In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.metrics import classification_report, cohen_kappa_score, accuracy_score, mean_absolute_error
from sklearn.utils.class_weight import compute_class_weight
import os
import zipfile

# Đọc dữ liệu đã qua tiền xử lý (Notebook 02)
PROCESSED_DIR = '../data/processed/'
df_train = pd.read_parquet(PROCESSED_DIR + 'train_cleaned.parquet')
df_valid = pd.read_parquet(PROCESSED_DIR + 'valid_cleaned.parquet')
df_test = pd.read_parquet(PROCESSED_DIR + 'test_cleaned.parquet')

# Ép nhãn về int64 để tương thích với hàm Loss
df_train['label'] = df_train['label'].astype(np.int64)
df_valid['label'] = df_valid['label'].astype(np.int64)

# Chuyển thành HuggingFace Dataset
datasets = DatasetDict({
    'train': Dataset.from_pandas(df_train[['Sentence_Normalized', 'label']]),
    'valid': Dataset.from_pandas(df_valid[['Sentence_Normalized', 'label']]),
    'test': Dataset.from_pandas(df_test[['Sentence_Normalized']]) # Test không có nhãn
})
print(f"✅ Dữ liệu sẵn sàng! Train: {len(df_train)} | Valid: {len(df_valid)} | Test: {len(df_test)}")

✅ Dữ liệu sẵn sàng! Train: 54626 | Valid: 7310 | Test: 7286


In [2]:
MODEL_NAME = "aubmindlab/bert-base-arabertv02"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(
        examples["Sentence_Normalized"], 
        padding="max_length", 
        truncation=True, 
        max_length=128  # Ép xuống 128 giúp chạy cực êm
    )

print("⏳ Đang Tokenize bằng AraBERTv2...")
tokenized_datasets = datasets.map(tokenize_function, batched=True)

# Format PyTorch Tensors
tokenized_datasets["train"].set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
tokenized_datasets["valid"].set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
tokenized_datasets["test"].set_format(type='torch', columns=['input_ids', 'attention_mask'])
print("✅ Hoàn tất Tokenize!")

⏳ Đang Tokenize bằng AraBERTv2...


Map:   0%|          | 0/54626 [00:00<?, ? examples/s]

Map:   0%|          | 0/7310 [00:00<?, ? examples/s]

Map:   0%|          | 0/7286 [00:00<?, ? examples/s]

✅ Hoàn tất Tokenize!


In [3]:
print("⚙️ KHỞI TẠO MÔ HÌNH VÀ LORA CHO MAE REGRESSION...")

# 🎯 ĐIỂM QUAN TRỌNG: num_labels = 1 cho Hồi quy (Regression)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=1
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["query", "value", "dense"] 
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

⚙️ KHỞI TẠO MÔ HÌNH VÀ LORA CHO MAE REGRESSION...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at aubmindlab/bert-base-arabertv02 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 2,384,641 || all params: 137,578,754 || trainable%: 1.7333


In [4]:
class MAETrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # 🎯 Ép labels sang float để tương thích hàm Hồi quy
        labels = inputs.pop("labels").float()
        outputs = model(**inputs)
        
        # Squeeze logits từ [batch_size, 1] về [batch_size]
        logits = outputs.logits.squeeze(-1)
        
        # Tính L1 Loss (MAE)
        loss = F.l1_loss(logits, labels)
            
        return (loss, outputs) if return_outputs else loss

def compute_metrics_mae(eval_pred):
    logits, labels = eval_pred
    logits = logits.squeeze(-1)
    
    # 🎯 Làm tròn số thực thành nhãn nguyên và giới hạn trong [0, 18]
    pred_labels = np.clip(np.round(logits), 0, 18).astype(int)
    
    qwk = cohen_kappa_score(labels, pred_labels, weights='quadratic')
    acc = accuracy_score(labels, pred_labels)
    
    return {"qwk": qwk, "accuracy": acc}

In [5]:
training_args = TrainingArguments(
    output_dir="../saved_models/arabert_lora_mae",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-4,               
    per_device_train_batch_size=8,    
    gradient_accumulation_steps=4,    
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="qwk",
    greater_is_better=True,
    bf16=True,
    fp16=False,  
    seed=42
)

trainer = MAETrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["valid"],
    compute_metrics=compute_metrics_mae,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("🚀 BẮT ĐẦU HUẤN LUYỆN MAE REGRESSION...")
trainer.train()

trainer.save_model("../saved_models/arabert_lora_mae_best")
tokenizer.save_pretrained("../saved_models/arabert_lora_mae_best")

🚀 BẮT ĐẦU HUẤN LUYỆN MAE REGRESSION...


  0%|          | 0/8535 [00:00<?, ?it/s]

{'loss': 1.9339, 'grad_norm': 29.786298751831055, 'learning_rate': 0.00028242530755711773, 'epoch': 0.29}
{'loss': 1.5569, 'grad_norm': 39.960174560546875, 'learning_rate': 0.00026485061511423544, 'epoch': 0.59}
{'loss': 1.4255, 'grad_norm': 33.939701080322266, 'learning_rate': 0.00024727592267135325, 'epoch': 0.88}


  0%|          | 0/457 [00:00<?, ?it/s]

{'eval_loss': 1.3933545351028442, 'eval_qwk': 0.7907377927145616, 'eval_accuracy': 0.3140902872777018, 'eval_runtime': 30.3643, 'eval_samples_per_second': 240.743, 'eval_steps_per_second': 15.051, 'epoch': 1.0}
{'loss': 1.3386, 'grad_norm': 35.69588088989258, 'learning_rate': 0.000229701230228471, 'epoch': 1.17}
{'loss': 1.2464, 'grad_norm': 32.765838623046875, 'learning_rate': 0.00021212653778558875, 'epoch': 1.46}
{'loss': 1.229, 'grad_norm': 20.345069885253906, 'learning_rate': 0.00019455184534270648, 'epoch': 1.76}


...

  0%|          | 0/457 [00:00<?, ?it/s]

{'eval_loss': 1.1596052646636963, 'eval_qwk': 0.8228063379117738, 'eval_accuracy': 0.4785225718194254, 'eval_runtime': 30.303, 'eval_samples_per_second': 241.23, 'eval_steps_per_second': 15.081, 'epoch': 4.0}
{'loss': 0.9715, 'grad_norm': 17.761323928833008, 'learning_rate': 5.39543057996485e-05, 'epoch': 4.1}
{'loss': 0.9255, 'grad_norm': 18.750051498413086, 'learning_rate': 3.6379613356766254e-05, 'epoch': 4.39}
{'loss': 0.9416, 'grad_norm': 22.873624801635742, 'learning_rate': 1.8804920913884008e-05, 'epoch': 4.69}
{'loss': 0.9078, 'grad_norm': 20.608800888061523, 'learning_rate': 1.2302284710017573e-06, 'epoch': 4.98}


  0%|          | 0/457 [00:00<?, ?it/s]

{'eval_loss': 1.1502023935317993, 'eval_qwk': 0.8199042198365971, 'eval_accuracy': 0.4897400820793434, 'eval_runtime': 29.9914, 'eval_samples_per_second': 243.736, 'eval_steps_per_second': 15.238, 'epoch': 5.0}
{'train_runtime': 3419.0695, 'train_samples_per_second': 79.884, 'train_steps_per_second': 2.496, 'train_loss': 1.170185503431699, 'epoch': 5.0}


('../saved_models/arabert_lora_mae_best\\tokenizer_config.json',
 '../saved_models/arabert_lora_mae_best\\special_tokens_map.json',
 '../saved_models/arabert_lora_mae_best\\vocab.txt',
 '../saved_models/arabert_lora_mae_best\\added_tokens.json',
 '../saved_models/arabert_lora_mae_best\\tokenizer.json')

In [6]:
print("🔍 ĐANG DỰ ĐOÁN TRÊN TẬP VALIDATION (MAE REGRESSION)...")
predictions_output = trainer.predict(tokenized_datasets["valid"])
logits = predictions_output.predictions.squeeze(-1) # Logits liên tục
true_labels = predictions_output.label_ids.astype(int)

# 🎯 Đưa số thực về nhãn 0-18
final_pred_labels = np.clip(np.round(logits), 0, 18).astype(int)

print("\n📊 === BÁO CÁO F1-SCORE MAE REGRESSION ===")
target_names = [f"Level_{i+1}" for i in range(19)]
print(classification_report(true_labels, final_pred_labels, target_names=target_names, zero_division=0))

print("\n🏆 === CÁC CHỈ SỐ METRIC BAREC ===")
print(f"🔸 QWK (Main Metric)       : {cohen_kappa_score(true_labels, final_pred_labels, weights='quadratic'):.4f}")
print(f"🔸 Acc19 (Exact Match)     : {accuracy_score(true_labels, final_pred_labels):.4f}")
print(f"🔸 Adjacent Acc (±1 Level)  : {np.mean(np.abs(true_labels - final_pred_labels) <= 1):.4f}")
print(f"🔸 Avg Distance (MAE)      : {mean_absolute_error(true_labels, final_pred_labels):.4f}")

🔍 ĐANG DỰ ĐOÁN TRÊN TẬP VALIDATION (MAE REGRESSION)...


  0%|          | 0/457 [00:00<?, ?it/s]


📊 === BÁO CÁO F1-SCORE MAE REGRESSION ===
              precision    recall  f1-score   support

     Level_1       0.71      0.55      0.62        44
     Level_2       0.35      0.12      0.18        68
     Level_3       0.47      0.45      0.46       182
     Level_4       0.11      0.22      0.15        78
     Level_5       0.54      0.49      0.51       417
     Level_6       0.19      0.21      0.20       189
     Level_7       0.54      0.62      0.57       701
     Level_8       0.59      0.60      0.59       613
     Level_9       0.38      0.48      0.42       236
    Level_10       0.63      0.74      0.68      1012
    Level_11       0.19      0.22      0.20       409
    Level_12       0.51      0.42      0.46      1491
    Level_13       0.25      0.36      0.29       349
    Level_14       0.57      0.54      0.55      1072
    Level_15       0.23      0.14      0.17       258
    Level_16       0.16      0.10      0.12       114
    Level_17       0.00      0.00     